In [1]:
import os
from typing import Annotated, TypedDict, Literal
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage, AIMessage, SystemMessage
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages

load_dotenv()

True

In [2]:
# State & LLM

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0.3)

class TuVanMuaHangState(TypedDict):
    messages: Annotated[list, add_messages]
    giai_doan: str
    san_pham_chinh: str
    so_lan_tuong_tac: int

In [3]:
# Node 1 - Phân tích giai đoạn mua hàng
def phan_tich_giai_doan(state: TuVanMuaHangState) -> dict:
    lich_su_ngan_gon = ""
    for msg in state["messages"][-6:]:
        role = "Khách" if isinstance(msg, HumanMessage) else "Bot"
        lich_su_ngan_gon += f"{role}: {msg.content}\n"
    
    prompt_phan_tich = f"""Dựa vào đoạn hội thoại này giữa khách hàng và shop bán đồ gia dụng:

{lich_su_ngan_gon}

Xác định khách đang ở giai đoạn nào:
- "tim_hieu": Khách mới bắt đầu tìm hiểu, hỏi chung chung, chưa có sản phẩm cụ thể trong đầu
- "so_sanh": Khách đã biết muốn gì, đang so sánh các lựa chọn, hỏi về tính năng/giá/thương hiệu
- "sap_mua": Khách gần như đã quyết định, hỏi về khuyến mãi, ship, bảo hành, cách đặt hàng

Chỉ trả về đúng một trong ba từ: tim_hieu, so_sanh, hoặc sap_mua"""

    ket_qua = llm.invoke([HumanMessage(content=prompt_phan_tich)])
    giai_doan = ket_qua.content.strip().lower()

    if giai_doan not in ["tim_hieu", "so_sanh", "sap_mua"]:
        giai_doan = "tim_hieu"

    so_lan = state.get("so_lan_tuong_tac", 0) + 1

    return {
        "giai_doan": giai_doan,
        "so_lan_tuong_tac": so_lan
    }

In [4]:
# Conditional Edge
def dinh_tuyen_theo_giai_doan(state: TuVanMuaHangState) -> str:
    giai_doan = state.get("giai_doan", "tim_hieu")

    routing = {
        "tim_hieu": "gioi_thieu_san_pham",
        "so_sanh": "ho_tro_so_sanh",
        "sap_mua": "chot_sale"
    }
    return routing.get(giai_doan, "gioi_thieu_san_pham")

In [5]:
# Node 2, 3, 4 - Ba nhánh tư vấn
def gioi_thieu_san_pham(state: TuVanMuaHangState) -> dict:
    """Nhánh 1: Khách đang tìm hiểu — giới thiệu tổng quan, gợi mở nhu cầu."""
    
    system = """Bạn là tư vấn viên của shop HOMELAB - đồ gia dụng thông minh tại Việt Nam.
    
Khách đang ở giai đoạn TÌM HIỂU — họ chưa rõ muốn gì. Nhiệm vụ của bạn:
- Hỏi thêm về nhu cầu cụ thể của khách (không gian, ngân sách, mục đích sử dụng)
- Giới thiệu tổng quan các dòng sản phẩm phù hợp
- Tạo cảm hứng và sự tin tưởng — đừng push bán ngay
Giọng văn: thân thiện, tự nhiên, như người tư vấn thật."""
    
    phan_hoi = llm.invoke([SystemMessage(content=system)] + state["messages"])
    return {"messages": [phan_hoi]}

def ho_tro_so_sanh(state: TuVanMuaHangState) -> dict:
    """Nhánh 2: Khách đang so sánh — cung cấp thông tin chi tiết, highlight điểm mạnh."""
    
    system = """Bạn là tư vấn viên của shop HOMELAB - đồ gia dụng thông minh tại Việt Nam.
    
Khách đang SO SÁNH các lựa chọn — họ muốn thông tin chi tiết để quyết định. Nhiệm vụ của bạn:
- So sánh trực tiếp, rõ ràng: tính năng, giá, bảo hành, thương hiệu
- Nêu bật điểm mạnh phù hợp với nhu cầu khách đã đề cập
- Đưa ra gợi ý cụ thể kèm lý do — đừng để khách tự quyết một mình
- Nếu cần, dùng format ngắn gọn kiểu "Sản phẩm A: ... | Sản phẩm B: ..."
Giọng văn: chuyên nghiệp, tự tin, có số liệu cụ thể."""

    phan_hoi = llm.invoke([SystemMessage(content=system)] + state["messages"])
    return {"messages": [phan_hoi]}

def chot_sale(state: TuVanMuaHangState) -> dict:
    """Nhánh 3: Khách gần mua — hỗ trợ chốt, cung cấp thông tin đặt hàng."""
    
    so_lan = state.get("so_lan_tuong_tac", 1)
    
    # Nếu đây là lần tương tác thứ 3 trở lên và khách sắp mua — offer ưu đãi
    uu_dai = ""
    if so_lan >= 3:
        uu_dai = "\nLưu ý: Khách đã tương tác nhiều lần — có thể đề xuất ưu đãi đặc biệt: giảm 5% hoặc freeship nếu đặt hôm nay."
    
    system = f"""Bạn là tư vấn viên của shop HOMELAB - đồ gia dụng thông minh tại Việt Nam.
    
Khách gần như ĐÃ QUYẾT ĐỊNH mua — họ chỉ cần một cú đẩy cuối. Nhiệm vụ của bạn:
- Xác nhận lại sản phẩm khách chọn, tóm tắt lợi ích chính
- Thông báo rõ: thời gian giao hàng, chính sách bảo hành, cách đặt hàng
- Tạo urgency nếu phù hợp (hàng sắp hết, ưu đãi có thời hạn)
- Hỏi địa chỉ giao hàng hoặc hướng dẫn bước tiếp theo cụ thể
Giọng văn: nhiệt tình, quyết đoán, nhưng không áp lực.{uu_dai}"""

    phan_hoi = llm.invoke([SystemMessage(content=system)] + state["messages"])
    return {"messages": [phan_hoi]}

In [6]:
# Xay graph
graph_builder = StateGraph(TuVanMuaHangState)

# Đăng ký tất cả Node
graph_builder.add_node("phan_tich",          phan_tich_giai_doan)
graph_builder.add_node("gioi_thieu_san_pham", gioi_thieu_san_pham)
graph_builder.add_node("ho_tro_so_sanh",      ho_tro_so_sanh)
graph_builder.add_node("chot_sale",           chot_sale)

# Edges: START → phân tích → [rẽ nhánh] → phản hồi → END
graph_builder.add_edge(START, "phan_tich")

graph_builder.add_conditional_edges(
    "phan_tich",              # Từ Node phân tích
    dinh_tuyen_theo_giai_doan, # Dùng hàm này để quyết định
    {                          # Map kết quả → Node tiếp theo
        "gioi_thieu_san_pham": "gioi_thieu_san_pham",
        "ho_tro_so_sanh":      "ho_tro_so_sanh",
        "chot_sale":           "chot_sale"
    }
)

# Tất cả các nhánh đều kết thúc tại END
graph_builder.add_edge("gioi_thieu_san_pham", END)
graph_builder.add_edge("ho_tro_so_sanh",      END)
graph_builder.add_edge("chot_sale",           END)

In [7]:
# Compile
graph = graph_builder.compile()

In [9]:
# Hàm chat wrapper cho Graph
def chat_with_graph(user_message, lich_su):

    messages = []
    for role, content in lich_su:
        if role == "user":
            messages.append(HumanMessage(content=content))
        else:
            messages.append(AIMessage(content=content))

    messages.append(HumanMessage(content=user_message))

    state_input = {
        "messages": messages,
        "giai_doan": "",
        "san_pham_chinh": "",
        "so_lan_tuong_tac": len(lich_su) // 2
    }

    result = graph.invoke(state_input)

    reply = result["messages"][-1].content

    new_history = lich_su + [
        ("user", user_message),
        ("assistant", reply)
    ]

    return reply, new_history

In [ ]:
# Kết nối với Gradio
import gradio as gr

def gradio_chat(user_message, history):

    # Convert sang format backend
    lich_su = []
    for user, bot in history:
        lich_su.append(("user", user))
        lich_su.append(("assistant", bot))

    # Gọi graph
    response, new_history = chat_with_graph(user_message, lich_su)

    # Convert lại cho Gradio
    gradio_history = []
    for i in range(0, len(new_history), 2):
        user_msg = new_history[i][1]
        bot_msg = new_history[i+1][1] if i+1 < len(new_history) else ""
        gradio_history.append((user_msg, bot_msg))

    return "", gradio_history
    
with gr.Blocks(theme=gr.themes.Soft()) as demo:
    gr.Markdown("## 🏠 HOMELAB - Tư vấn đồ gia dụng thông minh")

    chatbot = gr.Chatbot(height=500)

    msg = gr.Textbox(
        placeholder="Bạn đang cần tìm thiết bị gì cho gia đình?",
        show_label=False
    )

    clear = gr.Button("🗑️ Xóa chat")

    msg.submit(
        gradio_chat,
        inputs=[msg, chatbot],
        outputs=[msg, chatbot]
    )

    clear.click(lambda: [], None, chatbot)

demo.launch()